# Function 1 - Data Acquisition and Cleaning
## Hekamanu, Jean-Charles

In [ ]:
# Import all the nessary libraries
import re
import nltk
from nltk.corpus import stopwords, wordnet, words
from nltk.stem import WordNetLemmatizer
import pandas as pd
from textblob import TextBlob
from IPython.display import FileLink
import gc

In [ ]:
# download important corpus
import nltk
nltk.download('stopwords')
nltk.download('words')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()
english_words = set(words.words())

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package words to /root/nltk_data...
[nltk_data]   Unzipping corpora/words.zip.


In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/T-Nawaz/CSCE5300_BigData_Term_Project/refs/heads/main/data/covid19_tweets.csv", encoding='latin-1')

# Define original column order
original_columns = ['user_name', 'user_location', 'user_description', 'user_followers', 'user_friends', 'user_favourites', 'user_verified', 'date', 'text', 'hashtags', 'source', 'is_retweet']

# Reorder the column to match orginal dataset
df = df[original_columns]
print("Column Names:", df.columns.tolist())
print("Missing Values:\n", df.isnull().sum())

# Convert data types
df['date'] = pd.to_datetime(df['date'])
df['user_followers'] = pd.to_numeric(df['user_followers'], errors='coerce')

In [ ]:
# noise Removal Function
def noise_removal(text):
    text = str(text).lower()  # Ensure text is string
    text = re.sub(r'cov[\s\-]?id[\s\-]?19', 'covid-19', text)
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'\@\w+|\#', '', text)
    text = re.sub(r'[^a-zA-Z0-9\s\-]', '', text)
    return text

# non-english word Filtering Function
def is_english_text(text, threshold=0.5):
    tokens = str(text).lower().split()
    if not tokens:
        return False
    matches = sum(1 for word in tokens if word in english_words)
    return (matches / len(tokens)) >= threshold

In [ ]:

import nltk
from nltk.stem import WordNetLemmatizer

#Lemmatization function
def clean_tweet_lemmatized_keep_numbers(text):

    lemmatizer = WordNetLemmatizer()
    tokens = str(text).split()
    tokens = [word for word in tokens if word not in stop_words]
    tokens = [lemmatizer.lemmatize(word, pos='v') for word in tokens]
    return " ".join(tokens)

#process the data preprocessing
def full_cleaning_pipeline(text):
    cleaned = noise_removal(text)
    return clean_tweet_lemmatized_keep_numbers(cleaned)

In [ ]:
#Create a polarity score based on tweets
def get_sentiment(text):
    analysis = TextBlob(str(text))
    polarity = analysis.sentiment.polarity
    if polarity > 0:
        return 'positive'
    elif polarity < 0:
        return 'negative'
    else:
        return 'neutral'

In [ ]:
#Download the nltk corpus
import nltk
nltk.download('wordnet')

In [ ]:
df = df[df['user_description'].notnull()]
df = df[df['user_description'].str.strip() != '']
df = df[df['user_followers'] >= 200]
df = df[df['date'] >= '2019-01-01']


df = df[df['text'].apply(is_english_text)]


df['clean_text'] = df['text'].apply(full_cleaning_pipeline)

In [ ]:
df = df[df['clean_text'].apply(is_english_text)]

# Apply sentiment analysis
df['sentiment'] = df['clean_text'].apply(get_sentiment)


df = df.reset_index(drop=True)

# reorder columns to match original in addition to new columns at the end
final_columns = original_columns + ['clean_text', 'sentiment']
df = df[final_columns]

#  inspect the data and output the first 5 rows for checking
print(f"Filtered DataFrame size: {len(df)}")
print("First 5 rows of processed DataFrame:\n", df.head())

# save to CSV
df.to_csv('cleaned_covid19_tweetsv3.csv', index=True, encoding='utf-8-sig')

# Provide Download Link
FileLink('cleaned_covid19_tweetsv3.csv')

# Function 2 - Exploratory Data Analysis (EDA)
## Nalabothu, Murari

In [ ]:
#importing libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import ast

In [ ]:
# Load the dataset
df = pd.read_csv("cleaned_covid19_tweetsv3.csv", encoding='ISO-8859-1')

# --- Preprocessing ---
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df['tweet_length'] = df['clean_text'].astype(str).apply(len)
df['hashtags'] = df['hashtags'].apply(lambda x: ast.literal_eval(x) if pd.notnull(x) else [])
df.head()

In [ ]:
# --- 1. Tweet Volume Over Time ---
# Tracking daily tweet counts to visualize COVID-19 discussion spikes.
plt.figure(figsize=(12, 4))
df['date'].dt.date.value_counts().sort_index().plot()
plt.title("Tweet Volume Over Time")
plt.xlabel("Date")
plt.ylabel("Tweet Count")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# --- 2. Sentiment Distribution ---
# Bar plot showing positive, neutral, and negative tweet counts.
sns.countplot(data=df, x='sentiment', palette='Set2')
plt.title("Sentiment Distribution")
plt.show()

In [ ]:
# --- 3. Tweet Length Histogram ---
# Analyze how verbose users are.
sns.histplot(df['tweet_length'], bins=30, kde=True)
plt.title("Distribution of Tweet Length")
plt.xlabel("Number of Characters")
plt.show()

In [ ]:
# --- 4. Top Hashtags ---
# Extract and ranking top 15 hashtags by frequency.
from collections import Counter
hashtag_counter = Counter([tag.lower() for sublist in df['hashtags'] for tag in sublist])
top_hashtags = pd.DataFrame(hashtag_counter.most_common(15), columns=['Hashtag', 'Count'])

sns.barplot(data=top_hashtags, y='Hashtag', x='Count', palette='Blues_r')
plt.title("Top 15 Hashtags")
plt.show()

In [ ]:
# --- 5. Word Cloud ---
# Showing most common words (excluding hashtags/URLs).
from wordcloud import STOPWORDS
text = " ".join(df['clean_text'].dropna().astype(str))
wordcloud = WordCloud(width=1000, height=400, background_color='white',
                      stopwords=STOPWORDS).generate(text)

plt.figure(figsize=(14, 6))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title("Word Cloud of Tweet Content")
plt.show()

In [ ]:
# --- 6. Retweet Ratio ---
# How many tweets are original vs. retweets.
sns.countplot(x='is_retweet', data=df)
plt.title("Original Tweets vs Retweets")
plt.xticks([0, 1], ['Original', 'Retweet'])
plt.show()
# Since retweets are zero, we are analyzing only original tweets. This means the data reflects users’ own thoughts, reactions, and sentiments, rather than reshared or amplified content from others.”


In [ ]:
# --- 7. Top User Locations  ---
# bar chart of locations where tweets are coming from.
top_locs = df['user_location'].value_counts().head(10)
top_locs.plot(kind='barh', color='teal')
plt.title("Top 10 User Locations")
plt.xlabel("Number of Tweets")
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# --- 8. Follower vs Friend Scatter ---
# Distributions or scatterplot of social reach.
plt.figure(figsize=(6, 4))
sns.scatterplot(data=df, x='user_followers', y='user_friends', alpha=0.5)
plt.title("User Reach: Followers vs Friends")
plt.xscale('log')
plt.yscale('log')
plt.xlabel("Followers")
plt.ylabel("Friends")
plt.tight_layout()
plt.show()

# Function 3 - Feature Engineering and Selection
## Kumari, Kiran

In [ ]:
from pyspark.sql import SparkSession

#creating a spark session to import pyspark libraries
spark = SparkSession.builder.appName("COVID19_Tweet_Analysis").getOrCreate()

In [ ]:
from pyspark.ml.feature import Tokenizer, HashingTF, IDF, NGram
from pyspark.sql import SparkSession

# Possible Modification
# df = df.drop('text',axis=1)

# Convert pandas DataFrame to PySpark DataFrame
spark_df = spark.createDataFrame(df)

# Tokenize tweets
tokenizer = Tokenizer(inputCol="clean_text", outputCol="words")
words_data = tokenizer.transform(spark_df)

In [ ]:
def build_ngram_tfidf(tokenized_df, n=1, num_features=1000):
  """
  Generates TF-IDF vectors based on n-gram tokens.

  Parameters:
      tokenized_df (DataFrame): Tokenized Spark DataFrame
      n (int): n-gram level (1=unigram, 2=bigram, ...)
      num_features (int): TF-IDF matrix size

  Returns:
      DataFrame: Spark DataFrame with TF-IDF feature column 'features'
  """

  # Generate n-grams
  if n > 1:
      ngram = NGram(n=n, inputCol="words", outputCol="ngrams")
      ngram_df = ngram.transform(tokenized_df)
      input_col = "ngrams"
  else:
      ngram_df = tokenized_df
      input_col = "words"

  # TF-IDF
  hashingTF = HashingTF(inputCol=input_col, outputCol="raw_features", numFeatures=num_features)
  featurized_df = hashingTF.transform(ngram_df)
  idf = IDF(inputCol="raw_features", outputCol="features")
  idf_model = idf.fit(featurized_df)
  tfidf_df = idf_model.transform(featurized_df)

  return tfidf_df

In [ ]:
def data_split(tfidf_df):
  # Split data (90% train, 10% test as per paper)
  train_df, test_df = tfidf_df.randomSplit([0.9, 0.1], seed=42)
  return train_df, test_df

# Function 4 - Core Algorithm Implementation
## Nawaz, Tanzim

In [ ]:
from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier, RandomForestClassifier
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.feature import StringIndexer

from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
import numpy as np
import gc

In [ ]:
def train_svm(train_data, c_values=[0.1, 1, 10, 100]):
  """
  Train Linear Support Vector Machine (LinearSVC) model with hyperparameter tuning using stratified cross-validation.

  Parameters:
      train_data (pyspark.sql.DataFrame): Training dataset containing 'sentiment' (labels)
                                          and 'features' (TF-IDF vectors)
      c_values (list): List of regularization strengths to try (C). Smaller values imply stronger regularization.
                        Default: [0.1, 1.0, 10.0]

  Returns:
      svm_model (sklearn.svm.LinearSVC): Best LinearSVC model trained with optimal C value.
                                          Use .predict() for predictions on test data
      svm_cv_scores (list): Cross-validation accuracy scores for each C value
      best_c (float): Optimal C value that achieved highest cross-validation accuracy
  """

  # Convert Spark DataFrame to pandas for sklearn compatibility
  train_pd = train_data.toPandas()

  # Extract features and labels
  X_train = np.array([np.array(row) for row in train_pd['features']])
  y_train = train_pd['sentiment'].values

  # Clear pandas dataframe to free memory
  del train_pd
  gc.collect()

  # Stratified 10-fold cross-validation
  skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

  # Grid search for SVM
  best_c = c_values[0]
  best_cv_score = 0
  svm_cv_scores = []

  for c in c_values:
    svm = LinearSVC(C=c, max_iter=3000, random_state=42, dual=False)
    cv_scores = cross_val_score(svm, X_train, y_train, cv=skf, scoring='accuracy', n_jobs=-1)
    avg_cv_score = np.mean(cv_scores)
    svm_cv_scores.append(cv_scores)

    if avg_cv_score > best_cv_score:
      best_cv_score = avg_cv_score
      best_c = c

  # To free memory
  del skf
  gc.collect()

  # Train final model with best C
  svm_model = LinearSVC(C=best_c, max_iter=5000, random_state=42)
  svm_model.fit(X_train, y_train)

  # To free memory
  del X_train, y_train, cv_scores, svm, best_c
  gc.collect()

  return svm_model, svm_cv_scores

In [ ]:
def train_decision_tree(train_data, maxDepth=[5, 10, 15], minInstancesPerNode=[1, 5, 10]):
  """
  Train Decision Tree model with hyperparameter tuning using cross-validation

  Parameters:
      train_data (pyspark.sql.DataFrame): Training dataset containing 'sentiment' (labels)
                                        and 'features' (feature vectors)
      maxDepth (list): Maximum depth of the tree for grid search. Controls tree complexity.
                      Deeper trees can model more complex patterns but may overfit
      minInstancesPerNode (list): Minimum number of instances required at each leaf node.
                                Higher values prevent overfitting by requiring more samples per leaf

  Returns:
      dt_model (pyspark.ml.tuning.CrossValidatorModel): Best Decision Tree model from cross-validation.
                                                        Use .transform() for predictions on test data
      dt_cv_metrics (list): Cross-validation accuracy scores for all parameter combinations
  """
  evaluator = MulticlassClassificationEvaluator(
      labelCol="indexedSentiment", predictionCol="prediction", metricName="accuracy"
  )

  indexer = StringIndexer(inputCol="sentiment", outputCol="indexedSentiment")
  indexed_train_data = indexer.fit(train_data).transform(train_data)

  dt = DecisionTreeClassifier(labelCol="indexedSentiment", featuresCol="features")

  param_grid = ParamGridBuilder() \
      .addGrid(dt.maxDepth, maxDepth) \
      .addGrid(dt.minInstancesPerNode, minInstancesPerNode) \
      .build()

  cv = CrossValidator(
      estimator=dt,
      estimatorParamMaps=param_grid,
      evaluator=evaluator,
      numFolds=10,
      seed=42
  )

  dt_model = cv.fit(indexed_train_data)

  return dt_model, dt_model.avgMetrics

In [ ]:
def train_random_forest(train_data, numTrees=[50, 100, 150], maxDepth=[5, 10, 15]):
  """
  Train Random Forest model with hyperparameter tuning using cross-validation

  Parameters:
      train_data (pyspark.sql.DataFrame): Training dataset containing 'sentiment' (labels)
                                        and 'features' (feature vectors)
      numTrees (list): Number of decision trees in the forest for grid search.
                      More trees generally improve performance but increase computational cost
      maxDepth (list): Maximum depth of each tree in the forest.
                      Controls individual tree complexity and overall model complexity

  Returns:
      rf_model (pyspark.ml.tuning.CrossValidatorModel): Best Random Forest model from cross-validation.
                                                        Use .transform() for predictions on test data
      rf_cv_metrics (list): Cross-validation accuracy scores for all parameter combinations
  """
  evaluator = MulticlassClassificationEvaluator(
      labelCol="indexedSentiment", predictionCol="prediction", metricName="accuracy"
  )

  # Index the sentiment column
  indexer = StringIndexer(inputCol="sentiment", outputCol="indexedSentiment")
  indexed_train_data = indexer.fit(train_data).transform(train_data)

  rf = RandomForestClassifier(labelCol="indexedSentiment", featuresCol="features")

  param_grid = ParamGridBuilder() \
      .addGrid(rf.numTrees, numTrees) \
      .addGrid(rf.maxDepth, maxDepth) \
      .build()

  cv = CrossValidator(
      estimator=rf,
      estimatorParamMaps=param_grid,
      evaluator=evaluator,
      numFolds=10,
      seed=42
  )

  rf_model = cv.fit(indexed_train_data)

  return rf_model, rf_model.avgMetrics

In [ ]:
def train_logistic_regression(train_data, regParam=[0.01, 0.1, 1.0], elasticNetParam=[0.0, 0.5, 1.0]):
  """
  Train Logistic Regression model with hyperparameter tuning using cross-validation

  Parameters:
      train_data (pyspark.sql.DataFrame): Training dataset containing 'sentiment' (labels)
                                        and 'features' (feature vectors)
      regParam (list): Regularization parameter values for grid search.
                      Controls model complexity and prevents overfitting
      elasticNetParam (list): ElasticNet mixing parameter for grid search.
                              0.0 = L2 penalty (Ridge), 1.0 = L1 penalty (Lasso), 0.5 = balanced mix

  Returns:
      lr_model (pyspark.ml.tuning.CrossValidatorModel): Best Logistic Regression model from cross-validation.
                                                        Use .transform() for predictions on test data
      lr_cv_metrics (list): Cross-validation accuracy scores for all parameter combinations
  """
  evaluator = MulticlassClassificationEvaluator(
      labelCol="indexedSentiment", predictionCol="prediction", metricName="accuracy"
  )

  # Index the sentiment column
  indexer = StringIndexer(inputCol="sentiment", outputCol="indexedSentiment")
  indexed_train_data = indexer.fit(train_data).transform(train_data)

  lr = LogisticRegression(labelCol="indexedSentiment", featuresCol="features")

  param_grid = ParamGridBuilder() \
      .addGrid(lr.regParam, regParam) \
      .addGrid(lr.elasticNetParam, elasticNetParam) \
      .build()

  cv = CrossValidator(
      estimator=lr,
      estimatorParamMaps=param_grid,
      evaluator=evaluator,
      numFolds=10,
      seed=42
  )

  lr_model = cv.fit(indexed_train_data)

  return lr_model, lr_model.avgMetrics

In [ ]:
def train_knn(train_data, k_values=[3, 5, 7, 9, 11]):
  """
  Train K-Nearest Neighbors (KNN) model with hyperparameter tuning using cross-validation

  Parameters:
      train_data (pyspark.sql.DataFrame): Training dataset containing 'sentiment' (labels)
                                        and 'features' (feature vectors)
      k_values (list): Number of neighbors to consider for grid search.
                      Lower k = more flexible model (may overfit), higher k = smoother boundaries

  Returns:
      knn_model (sklearn.neighbors.KNeighborsClassifier): Best KNN model trained with optimal k.
                                                          Use .predict() for predictions on test data
      knn_cv_scores (list): Cross-validation accuracy scores for each k value
      best_k (int): Optimal number of neighbors that achieved highest cross-validation accuracy
  """
  # Convert Spark DataFrame to pandas for sklearn compatibility
  train_pd = train_data.toPandas()

  # Extract features and labels
  X_train = np.array([np.array(row) for row in train_pd['features']])
  y_train = train_pd['sentiment'].values

  # Stratified 10-fold cross-validation
  skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

  # Grid search for KNN
  best_k = k_values[0]
  best_cv_score = 0
  knn_cv_scores = []

  for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    cv_scores = cross_val_score(knn, X_train, y_train, cv=skf, scoring='accuracy', n_jobs=-1)
    avg_cv_score = np.mean(cv_scores)
    knn_cv_scores.append(cv_scores)

    if avg_cv_score > best_cv_score:
      best_cv_score = avg_cv_score
      best_k = k

  # Train final model with best k
  knn_model = KNeighborsClassifier(n_neighbors=best_k)
  knn_model.fit(X_train, y_train)

  return knn_model, knn_cv_scores, best_k

In [ ]:
def predict_model(model, test_data):
    """
    Generate predictions using a trained classification model.

    This function automatically detects whether the model is a PySpark ML model or a scikit-learn model.

    Parameters:
        model (object): Trained model (PySpark or scikit-learn)
        test_data (DataFrame):
            - PySpark model: a Spark DataFrame with 'sentiment' and 'features' columns.
            - sklearn model: a Spark DataFrame that will be converted to pandas with 'features' and 'sentiment'

    Returns:
        dict: {
            'y_true': list or array of ground truth labels,
            'y_pred': list or array of predicted labels,
            'raw': Spark DataFrame or pandas DataFrame with predictions
        }
    """
    if hasattr(model, "transform"):  # PySpark model
        from pyspark.ml.feature import StringIndexer

        indexer = StringIndexer(inputCol="sentiment", outputCol="indexedSentiment")
        indexed_test = indexer.fit(test_data).transform(test_data)

        predictions = model.transform(indexed_test)

        y_true = [int(row["indexedSentiment"]) for row in predictions.select("indexedSentiment").collect()]
        y_pred = [int(row["prediction"]) for row in predictions.select("prediction").collect()]

        return {
            "y_true": y_true,
            "y_pred": y_pred,
            "raw": predictions
        }

    else:  # We assume scikit-learn model

        test_pd = test_data.toPandas()
        X_test = np.array([np.array(row) for row in test_pd["features"]])
        y_true = test_pd["sentiment"].values
        y_pred = model.predict(X_test)

        result_df = test_pd.copy()
        result_df["prediction"] = y_pred

        return {
            "y_true": y_true,
            "y_pred": y_pred,
            "raw": result_df
        }


# Live Demonstration
## Nawaz, Tanzim

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
print('Logistic Regression Model')
print(f"N-gram : 1 Matrix size : 3000")
tfidf_df = build_ngram_tfidf(words_data, n=1, num_features=1000)
train_df, test_df = data_split(tfidf_df)
LRmodel,LR_metrics = train_logistic_regression(train_df)

Logistic Regression Model
N-gram : 1 Matrix size : 3000


In [ ]:
print(f"Best regParam: {LRmodel.bestModel.getRegParam()}")
print(f"Best elasticNetParam: {LRmodel.bestModel.getElasticNetParam()}")

Best regParam: 0.01
Best elasticNetParam: 0.0


In [ ]:
lr_out = predict_model(LRmodel, test_df)
lr_out['y_true'][:5], lr_out['y_pred'][:5]

lr_accuracy = accuracy_score(lr_out['y_true'], lr_out['y_pred'])
lr_precision = precision_score(lr_out['y_true'], lr_out['y_pred'], average='weighted', zero_division=0)
lr_recall = recall_score(lr_out['y_true'], lr_out['y_pred'], average='weighted', zero_division=0)
lr_f1 = f1_score(lr_out['y_true'], lr_out['y_pred'], average='weighted', zero_division=0)

print({"accuracy", lr_accuracy, "precision", lr_precision, "recall", lr_recall, "f1", lr_f1})

{0.6778305423644089, 0.6749270203906068, 0.6714029678232903, 'precision', 'accuracy', 'f1', 'recall'}


# Function 5 - Experiment Simulation and Metrics Calculation
## Magunta, Sai Grishyanth

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [ ]:
lr_validation_data = [{}]
for i in range(1,5):
  for j in [1000,3000]:
    tfidf_df = build_ngram_tfidf(words_data, n=i, num_features=j)
    train_df, test_df = data_split(tfidf_df)
    LRmodel,LR_metrics = train_logistic_regression(train_df)
    lr_out = predict_model(LRmodel, test_df)
    lr_accuracy = accuracy_score(lr_out['y_true'], lr_out['y_pred'])
    lr_precision = precision_score(lr_out['y_true'], lr_out['y_pred'], average='weighted', zero_division=0)
    lr_recall = recall_score(lr_out['y_true'], lr_out['y_pred'], average='weighted', zero_division=0)
    lr_f1 = f1_score(lr_out['y_true'], lr_out['y_pred'], average='weighted', zero_division=0)
    lr_validation_data.append({"n_gram": i, "matrix_size": j,"accuracy": lr_accuracy, "precision": lr_precision, "recall": lr_recall, "f1": lr_f1})
    # print(lr_out['y_true'],lr_out['y_pred'])
    print(f"N-gram : {i} Matrix size : {j}")
    print(lr_accuracy,lr_precision,lr_recall,lr_f1)
    del LRmodel, LR_metrics, lr_out, lr_accuracy, lr_precision, lr_recall, lr_f1
    gc.collect()

N-gram : 1 Matrix size : 1000
0.6778305423644089 0.6749270203906068 0.6778305423644089 0.6714029678232903
N-gram : 1 Matrix size : 3000
0.7803049237690577 0.779209925017974 0.7803049237690577 0.7780535619242164


In [ ]:
dt_validation_data = [{}]
for i in range(1,5):
  for j in [1000,3000]:
    tfidf_df = build_ngram_tfidf(words_data, n=i, num_features=j)
    train_df, test_df = data_split(tfidf_df)
    DTmodel,DT_metrics = train_decision_tree(train_df)
    dt_out = predict_model(DTmodel, test_df)
    dt_accuracy = accuracy_score(dt_out['y_true'], dt_out['y_pred'])
    dt_precision = precision_score(dt_out['y_true'], dt_out['y_pred'], average='weighted', zero_division=0)
    dt_recall = recall_score(dt_out['y_true'], dt_out['y_pred'], average='weighted', zero_division=0)
    dt_f1 = f1_score(dt_out['y_true'], dt_out['y_pred'], average='weighted', zero_division=0)
    dt_validation_data.append({"n_gram": i, "matrix_size": j,"accuracy": dt_accuracy, "precision": dt_precision, "recall": dt_recall, "f1": dt_f1})
    # print(dt_out['y_true'], dt_out['y_pred'])
    print(f"N-gram : {i} Matrix size : {j}")
    print(dt_accuracy,dt_precision,dt_recall,dt_f1)
    del DTmodel, DT_metrics, dt_out, dt_accuracy, dt_precision, dt_recall, dt_f1
    gc.collect()

N-gram : 1 Matrix size : 1000
0.5989752561859535 0.6549077355432519 0.5989752561859535 0.5605815248148794
N-gram : 1 Matrix size : 3000
0.6197200699825044 0.7164928470683005 0.6197200699825044 0.5807331035460719
N-gram : 2 Matrix size : 1000
0.4530117470632342 0.5196438426706267 0.4530117470632342 0.37772493235615484
N-gram : 2 Matrix size : 3000
0.45763559110222446 0.5794056635938608 0.45763559110222446 0.3636943749950196
N-gram : 3 Matrix size : 1000
0.42839290177455636 0.5259893902534675 0.42839290177455636 0.32874104237857293
N-gram : 3 Matrix size : 3000
0.42676830792301923 0.5654113361394769 0.42676830792301923 0.3064885719321116
N-gram : 4 Matrix size : 1000
0.4162709322669333 0.6884623101629488 0.4162709322669333 0.2513941874732548
N-gram : 4 Matrix size : 3000
0.41677080729817545 0.5316938299744489 0.41677080729817545 0.25667662141336256


In [ ]:
rf_validation_data = []
knn_validation_data = [{}]
svm_validation_data = [{}]

for i in range(2,5):
  for j in [1000,3000]:
    tfidf_df = build_ngram_tfidf(words_data, n=i, num_features=j)
    train_df, test_df = data_split(tfidf_df)
    RFmodel,RF_metrics = train_random_forest(train_df)
    rf_out = predict_model(RFmodel, test_df)
    rf_accuracy = accuracy_score(rf_out['y_true'], rf_out['y_pred'])
    rf_precision = precision_score(rf_out['y_true'], rf_out['y_pred'], average='weighted', zero_division=0)
    rf_recall = recall_score(rf_out['y_true'], rf_out['y_pred'], average='weighted', zero_division=0)
    rf_f1 = f1_score(rf_out['y_true'], rf_out['y_pred'], average='weighted', zero_division=0)
    rf_validation_data.append({"n_gram": i, "matrix_size": j,"accuracy": rf_accuracy, "precision": rf_precision, "recall": rf_recall, "f1": rf_f1})
    # print(rf_out['y_true'],rf_out['y_pred'])
    print('RF')
    print(f"N-gram : {i} Matrix size : {j}")
    print(rf_accuracy,rf_precision,rf_recall,rf_f1)

    del RFmodel,RF_metrics,rf_out,rf_accuracy,rf_precision,rf_recall,rf_f1
    gc.collect()

    print('KNN')
    KNNmodel,knn_metrics,_ = train_knn(train_df)
    knn_out = predict_model(KNNmodel, test_df)
    knn_accuracy = accuracy_score(knn_out['y_true'], knn_out['y_pred'])
    knn_precision = precision_score(knn_out['y_true'], knn_out['y_pred'], average='weighted', zero_division=0)
    knn_recall = recall_score(knn_out['y_true'], knn_out['y_pred'], average='weighted', zero_division=0)
    knn_f1 = f1_score(knn_out['y_true'], knn_out['y_pred'], average='weighted', zero_division=0)
    knn_validation_data.append({"n_gram": i, "matrix_size": j,"accuracy": knn_accuracy, "precision": knn_precision, "recall": knn_recall, "f1": knn_f1})
    # print(knn_out['y_true'],knn_out['y_pred'])
    print(f"N-gram : {i} Matrix size : {j}")
    print(knn_accuracy,knn_precision,knn_recall,knn_f1)

    del KNNmodel,knn_metrics,_,knn_out,knn_accuracy,knn_precision,knn_recall,knn_f1
    gc.collect()

    del tfidf_df, train_df, test_df
    gc.collect()

RF
N-gram : 2 Matrix size : 1000
0.472381904523869 0.5686799679945921 0.472381904523869 0.4272336094814131
KNN
N-gram : 2 Matrix size : 1000
0.4443889027743064 0.4844220345771431 0.4443889027743064 0.39186410103850655


In [ ]:
rf_validation_data

In [ ]:
rf_validation_data

In [ ]:
knn_validation_data

In [ ]:
knn_validation_data

In [ ]:
svm_validation_data = [{}]
for i in range(1,5):
  for j in [1000,3000]:
    tfidf_df = build_ngram_tfidf(words_data, n=i, num_features=j)
    train_df, test_df = data_split(tfidf_df)
    SVMmodel,SVM_metrics = train_svm(train_df)
    svm_out = predict_model(SVMmodel, test_df)
    svm_accuracy = accuracy_score(svm_out['y_true'], svm_out['y_pred'])
    svm_precision = precision_score(svm_out['y_true'], svm_out['y_pred'], average='weighted', zero_division=0)
    svm_recall = recall_score(svm_out['y_true'], svm_out['y_pred'], average='weighted', zero_division=0)
    svm_f1 = f1_score(svm_out['y_true'], svm_out['y_pred'], average='weighted', zero_division=0)
    svm_validation_data.append({"n_gram": i, "matrix_size": j,"accuracy": svm_accuracy, "precision": svm_precision, "recall": svm_recall, "f1": svm_f1})
    # print(svm_out['y_true'],svm_out['y_pred'])
    print(f"N-gram : {i} Matrix size : {j}")
    print(svm_accuracy,svm_precision,svm_recall,svm_f1)
    del SVMmodel, SVM_metrics, svm_out, svm_accuracy, svm_precision, svm_recall, svm_f1
    gc.collect()

# Function 6 - Final Results Visualization
## Ravula, Vishal Suman

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import matplotlib as mpl

In [ ]:
# File Path
file_path = 'https://raw.githubusercontent.com/T-Nawaz/CSCE5300_BigData_Term_Project/refs/heads/main/data/model_metrics.csv'
df = pd.read_csv(file_path)

# Filter for SVM model data
for mdl in ['SVM','KNN','DT','RF','LR']:
  svm_df = df[df['model'] == mdl]

  # Font Options
  mpl.rcParams['font.family'] = 'serif'
  mpl.rcParams['font.serif'] = ['DejaVu Serif']
  mpl.rcParams.update({
      'font.size': 12,
      'axes.titlesize': 14,
      'axes.labelsize': 13,
      'legend.fontsize': 12,
      'xtick.labelsize': 11,
      'ytick.labelsize': 11
  })

  # Columns
  metrics = ['accuracy',	'precision',	'recall',	'f1']
  n_grams = sorted(svm_df['ngram'].unique())
  matrix_sizes = sorted(svm_df['matrix_size'].unique())
  bar_width = 0.1
  x = np.arange(len(n_grams))

  # Custom Colors
  colors = {
      'accuracy_1000': '#1f77b4',
      'accuracy_3000': '#aec7e8',
      'precision_1000': '#ff7f0e',
      'precision_3000': '#ffbb78',
      'recall_1000': '#2ca02c',
      'recall_3000': '#98df8a',
      'f1_1000': '#d62728',
      'f1_3000': '#ff9896'
  }

  # Bar Plot
  plt.figure(figsize=(14, 6))
  shift = [-1.5, -0.5, 0.5, 1.5]

  for i, metric in enumerate(metrics):
      for j, msize in enumerate(matrix_sizes):
          # Filter for the specific matrix size within the SVM data
          values = svm_df[svm_df['matrix_size'] == msize][metric].values
          label = f'{metric} - {msize}'
          color_key = f'{metric}_{msize}'
          positions = x + (shift[i] + 0.25 * j) * bar_width

          plt.bar(positions, values, width=bar_width, label=label, color=colors[color_key])

  # Labeling and Style
  plt.xticks(x, n_grams)
  plt.xlabel("NGram Value")
  plt.ylabel("Score")
  plt.ylim(0, 1.05)
  plt.title(f"{mdl} Evaluation Metrics across NGram and Matrix Size", fontweight='bold', fontsize=16)
  plt.grid(axis='y', linestyle='--', alpha=0.6)
  plt.legend(ncol=2, bbox_to_anchor=(1.02, 1), loc='upper left')

  plt.tight_layout()
  plt.show()

In [ ]:
df[(df['matrix_size'] == 3000) & ((df['model'] == 'SVM') | (df['model'] == 'LR'))]